## Calculating Width

When we perform a simulation, we can track the coherence width of the wavefunction


In [ ]:
from pathlib import Path
from typing import TYPE_CHECKING

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import HTML
from scipy.constants import Boltzmann  # type: ignore stubs
from slate_core import FundamentalBasis, TupleMetadata
from slate_core.metadata import (
    AxisDirections,
    Domain,
    spaced_volume_metadata_from_stacked_delta_x,
)
from slate_core.plot import (
    animate_data_over_list_1d_k,
    animate_data_over_list_1d_x,
    array_against_basis,
)
from slate_core.util import cached

from slate_quantum import operator, state
from slate_quantum.dynamics import (
    LangevinParameters,
    select_realization,
    solve_periodic_caldeira_leggett,
)
from slate_quantum.metadata import RepeatedMetadata, SpacedTimeMetadata

if TYPE_CHECKING:
    from slate_quantum.dynamics import (
        RealizationList,
        RealizationListBasis,
    )

metadata = spaced_volume_metadata_from_stacked_delta_x((np.array([2 * np.pi]),), (41,))

potential_single = operator.build.cos_potential(metadata, 2)
potential = operator.build.repeat_potential(potential_single, (3,))
potential_metadata = potential.basis.inner.outer_recast.metadata()


mass = 1
hamiltonian = operator.build.kinetic_hamiltonian(potential, mass, hbar=1)
initial_state = state.build_coherent(potential_metadata, (0,), (0,), sigma_0=(0.5,))


parameters = LangevinParameters(
    mass=mass, lambda_=0.5, temperature=8 / Boltzmann, hbar=1
)
times = FundamentalBasis(SpacedTimeMetadata(60, domain=Domain(delta=20 * np.pi)))


@cached(Path("data/caldeira_leggett_wavefunction.npz"))
def _simulate_quantum() -> RealizationList[
    RealizationListBasis[
        SpacedTimeMetadata, TupleMetadata[tuple[RepeatedMetadata, ...], AxisDirections]
    ]
]:

    return solve_periodic_caldeira_leggett(
        initial_state,
        times,
        parameters,
        potential,
        method="Rouchon",
        target_delta=1e-3,
    )


realizations = _simulate_quantum()
evolution = select_realization(realizations, 0)

normalization = state.normalization_each(evolution)
fig, ax, line = array_against_basis(normalization)

fig, ax, _anim0 = animate_data_over_list_1d_x(evolution, measure="abs")
ax.set_title("Real part of the state in x space")
display(HTML(_anim0.to_jshtml()))
plt.close(fig)


fig, ax, _anim0 = animate_data_over_list_1d_k(evolution, measure="abs")
ax.set_title("Real part of the state in k space")
display(HTML(_anim0.to_jshtml()))
plt.close(fig)

We can keep track of the width of the wavefunction directly


In [ ]:
from slate_core import Array, Basis, TupleBasis

from slate_quantum.dynamics import caldeira_leggett
from slate_quantum.dynamics.langevin import eval_delta_xp


@cached(Path("data/caldeira_leggett_positions.npz"))
def _simulate_quantum_uncertainty() -> tuple[
    Array[
        TupleBasis[tuple[FundamentalBasis, Basis[SpacedTimeMetadata]], None],
        np.dtype[np.floating],
    ],
    Array[
        TupleBasis[tuple[FundamentalBasis, Basis[SpacedTimeMetadata]], None],
        np.dtype[np.floating],
    ],
]:

    _, ratio, _ = caldeira_leggett.solve_periodic_locations(
        initial_state,
        times,
        parameters,
        potential,
        method="Rouchon",
        target_delta=1e-3,
    )
    return eval_delta_xp(parameters, ratio)  # type: ignore[aa]


delta_x, delta_p = _simulate_quantum_uncertainty()


fig, ax, line = array_against_basis(delta_x, measure="real")
ax.set_xlabel("Time / s")
ax.set_ylabel("Delta x / m")
ax.set_title("Delta x of the state")

fig, ax, line = array_against_basis(delta_p, measure="real")
ax.set_xlabel("Time / s")
ax.set_ylabel("Delta p / kg m/s")
ax.set_title("Delta p of the state")
